In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, GOLD, STEEL

Drag $b$: the red staircase is the sine after quantization to $2^b$ levels,
and the gold trace is the error left behind. The audio card underneath
always plays the current bit depth.

In [ ]:
# hide
# autorun
F0 = 220.0
t = np.linspace(0.0, 2 / F0, 900, endpoint=False)   # two cycles on screen
y = 0.95 * np.sin(2 * np.pi * F0 * t)
sr = 44100
seg = np.arange(sr) / sr                            # one second, for the ear
tone = 0.95 * np.sin(2 * np.pi * F0 * seg)

def quantize(x, b):
    # the chapter's convention: scale to the 2^b integer levels and floor
    L = 2 ** (b - 1) - 1
    return np.floor(L * x) / L

BITS = range(2, 13)
YQ = {b: quantize(y, b) for b in BITS}
ERR = {b: y - YQ[b] for b in BITS}
LEVELS = {}
for b in BITS:
    L = 2 ** (b - 1) - 1
    if b <= 4:                        # level guides only while they are legible
        xs, ys = [], []
        for k in range(-L, L + 1):
            xs += [0, 2000 / F0, None]
            ys += [k / L, k / L, None]
        LEVELS[b] = (xs, ys)
    else:
        LEVELS[b] = ([], [])

def figure():
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        row_heights=[0.68, 0.32], vertical_spacing=0.1)
    fig.add_scatter(x=LEVELS[3][0], y=LEVELS[3][1], mode="lines",
                    line=dict(color=STEEL, width=1, dash="dot"), row=1, col=1)
    fig.add_scatter(x=t * 1000, y=y, mode="lines",
                    line=dict(color=STEEL, width=1.6), row=1, col=1)
    fig.add_scatter(x=t * 1000, y=YQ[3], mode="lines",
                    line=dict(color=RED, width=2), row=1, col=1)
    fig.add_scatter(x=t * 1000, y=ERR[3], mode="lines",
                    line=dict(color=GOLD, width=1.6), row=2, col=1)
    fig.update_yaxes(range=[-1.15, 1.15], title_text="Amplitude",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[0, 1.05], title_text="Error",
                     fixedrange=True, row=2, col=1)
    fig.update_xaxes(fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, 2000 / F0], title_text="Time (ms)",
                     fixedrange=True, row=2, col=1)
    return fig

def controls(fig):
    b = widgets.IntSlider(description="Bit depth b", min=2, max=12, value=3)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(b, YQ=YQ, ERR=ERR, LEVELS=LEVELS, readout=readout):
        with fig.batch_update():
            fig.data[0].x, fig.data[0].y = LEVELS[b]
            fig.data[2].y = YQ[b]
            fig.data[3].y = ERR[b]
        L = 2 ** (b - 1) - 1
        readout.value = (f"<span style='font-size:0.9em'>2<sup>{b}</sup> = "
                         f"{2 ** b} levels &nbsp;·&nbsp; spacing 1/{L} "
                         f"≈ {1 / L:.4f}</span>")

    widgets.interactive_output(update, {"b": b})

    # the audio card under the controls always holds the current settings: a
    # change clears it and it re-renders when the pointer releases the
    # slider (keyboard nudges settle on a short timer instead). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def render(tone=tone, sr=sr, quantize=quantize):
        x = quantize(tone, b.value)
        x *= 0.125 / np.abs(x).max()          # about -18 dBFS, a safe level
        x[:441] *= np.linspace(0, 1, 441)
        x[-441:] *= np.linspace(1, 0, 441)
        out.outputs = ()
        out.append_display_data(Audio(x.astype(np.float32), rate=sr, normalize=False))

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        if not dirty:
            out.outputs = ()
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    b.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([b, readout, out, gate])

icm_plotly.show(figure, controls)